# Phase 7 — Prior Case Retrieval · Kaggle 2× T4

**Inputs:** `prefixlm-run1` (tokenizer) + `phase1-models`. Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN`.

**Yours (rule 17.5):** `pcr_model.py` (`build_pcr_model(cfg)` + `embed(ids [B,L]) -> [B,D]`, your pooling and adaptation
methods) and every `TODO` in the plan. **Harness:** data + chunking, negatives (random / BM25 / teacher), InfoNCE or pairwise
loss, in-batch negatives, KD from teacher similarity scores, dev Micro-F1@7 each epoch, best epoch, test top-7 → zip.

Technique list (doc 13.2.1): Full FT · Partial FT · QLoRA · **LoRMA** · Prefix · Prompt · IA³ · **KD** — at least 6 experiments,
5 different adaptation techniques + the KD experiment.

Order: cells 1–6 (tests pass, data check, SMOKE TEST PASSED) → `sum_model.py` + `pcr_model.py` → plan → model check → run.

In [ ]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
NB_START = time.time()
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

In [ ]:
# 2 — Phase-7 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P7 = {}
for split, d in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    P7[f"{split}_q"] = hf_hub_download(REPO, f"Phase_7_PCR/{d}/{split}.json", repo_type="dataset")
    P7[f"{split}_pool"] = hf_hub_download(REPO, f"Phase_7_PCR/{d}/{split}_candidates.json", repo_type="dataset")
print(json.dumps(P7, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")

In [ ]:
%%writefile pcr_data.py
"""pcr_data.py — Phase-7 prior case retrieval data handling."""
import hashlib, json, math, os, pickle
from collections import Counter

import numpy as np

ID_KEYS = ("id", "query_id", "case_id", "doc_id", "candidate_id", "qid", "cid", "name")
TEXT_KEYS = ("text", "sentences", "document", "content", "body", "judgment", "judgement")
REL_KEYS = ("relevant_candidates", "relevant", "relevant_cases", "positives", "labels", "gold")


def _get(rec, keys):
    for k in keys:
        if k in rec:
            return rec[k]
    return None


def _text(v):
    if v is None:
        return None
    if isinstance(v, (list, tuple)):
        return " ".join(str(x).strip() for x in v if str(x).strip())
    return str(v)


def _records(obj):
    if isinstance(obj, dict) and not any(k in obj for k in ID_KEYS):
        for k in ("data", "queries", "candidates", "cases", "examples"):
            if isinstance(obj.get(k), (list, dict)):
                return _records(obj[k])
        out = []
        for k, v in obj.items():
            if isinstance(v, dict):
                out.append(dict(v, **({} if _get(v, ID_KEYS) is not None else {"id": k})))
            else:
                out.append({"id": k, "text": v})
        return out
    return list(obj)


def _sid(x):
    return x if isinstance(x, str) else str(x)


def load_queries(path):
    out = {}
    for r in _records(json.load(open(path, encoding="utf-8"))):
        qid = _sid(_get(r, ID_KEYS))
        text = _text(_get(r, TEXT_KEYS))
        assert text is not None, f"query {qid}: no text field among {TEXT_KEYS}; keys {list(r)[:8]}"
        rel = _get(r, REL_KEYS)
        out[qid] = dict(id=qid, text=text, relevant=None if rel is None else [_sid(c) for c in rel])
    return out


def load_pool(path):
    out = {}
    for r in _records(json.load(open(path, encoding="utf-8"))):
        cid = _sid(_get(r, ID_KEYS))
        text = _text(_get(r, TEXT_KEYS))
        assert text is not None, f"candidate {cid}: no text field among {TEXT_KEYS}; keys {list(r)[:8]}"
        out[cid] = dict(id=cid, text=text)
    return out


def tokenize_texts(texts, encode_batch, cache_dir=None, tag=""):
    h = hashlib.md5((tag + "|".join(f"{k}:{len(v)}" for k, v in texts.items())).encode()).hexdigest()[:16]
    path = os.path.join(cache_dir, f"pcr_tok_{h}.pkl") if cache_dir else None
    if path and os.path.exists(path):
        return pickle.load(open(path, "rb"))
    keys = list(texts)
    out = {}
    for i in range(0, len(keys), 256):
        ks = keys[i:i + 256]
        for k, ids in zip(ks, encode_batch([texts[k] for k in ks])):
            out[k] = np.asarray(ids, dtype=np.uint16)
    if path:
        os.makedirs(cache_dir, exist_ok=True)
        tmp = path + f".tmp{os.getpid()}"
        pickle.dump(out, open(tmp, "wb"), protocol=4)
        os.replace(tmp, path)
    return out


def chunk_ids(ids, chunk_len, n_chunks, mode="first", stride=None):
    n, L = len(ids), int(chunk_len)
    if n == 0:
        return [np.zeros(1, np.uint16)]
    if n <= L:
        return [ids]
    k = max(1, int(n_chunks))
    st = int(stride or L)
    starts_all = list(range(0, n - L + 1, st))
    if starts_all[-1] != n - L and mode != "first":
        starts_all.append(n - L)
    if mode == "first":
        starts = starts_all[:k]
    elif mode == "spread":
        if k >= len(starts_all):
            starts = starts_all
        else:
            starts = sorted({starts_all[round(i * (len(starts_all) - 1) / max(1, k - 1))] for i in range(k)}) if k > 1 else [0]
    elif mode == "head_tail":
        h = (k + 1) // 2
        starts = sorted(set(starts_all[:h] + (starts_all[-(k - h):] if k - h else [])))
    else:
        raise ValueError(f"chunk_mode must be first | spread | head_tail, got {mode!r}")
    return [ids[s:s + L] for s in starts]


class BM25:

    def __init__(self, docs, k1=1.2, b=0.75, max_terms=None, vocab=65536):
        from scipy import sparse
        self.keys = list(docs)
        rows, cols, vals, lens = [], [], [], []
        for i, k in enumerate(self.keys):
            ids = docs[k] if max_terms is None else docs[k][:max_terms]
            t, c = np.unique(np.asarray(ids, np.int64), return_counts=True)
            rows.append(np.full(len(t), i)); cols.append(t); vals.append(c.astype(float)); lens.append(c.sum())
        N = len(self.keys)
        rows, cols, vals = (np.concatenate(x) if x else np.zeros(0) for x in (rows, cols, vals))
        tf = sparse.csr_matrix((vals, (rows.astype(np.int64), cols.astype(np.int64))), shape=(N, vocab))
        lens = np.asarray(lens, float)
        df = np.bincount(cols.astype(np.int64), minlength=vocab)
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
        norm = k1 * (1 - b + b * lens / max(1e-9, lens.mean() if N else 1.0))
        tf = tf.tocoo()
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * self.idf[tf.col]
        self.W = sparse.csr_matrix((w, (tf.row, tf.col)), shape=(N, vocab)).T.tocsr()

    def scores(self, q_ids, max_terms=None):
        q = np.unique(np.asarray(q_ids if max_terms is None else q_ids[:max_terms], np.int64))
        return np.asarray(self.W[q].sum(0)).ravel()


def rank_pool(scores, keys, exclude=(), k=None):
    order = np.argsort(-np.asarray(scores), kind="stable")
    ex = set(exclude)
    out = [keys[i] for i in order if keys[i] not in ex]
    return out if k is None else out[:k]


def micro_at_k(ranked, gold, k=7):
    tp = n_pred = n_gold = 0
    for q, g in gold.items():
        g = set(g)
        r = list(ranked.get(q, []))[:k]
        tp += len(set(r) & g)
        n_pred += k
        n_gold += len(g)
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r) if p + r else 0.0, tp=tp, n_queries=len(gold))


def recall_at(ranked, gold, k):
    tp = sum(len(set(list(ranked.get(q, []))[:k]) & set(g)) for q, g in gold.items())
    return tp / max(1, sum(len(g) for g in gold.values()))


def validate_prediction(pred, queries, pool, k=7):
    assert isinstance(pred, dict), "prediction.json must be a JSON object {query_id: [candidate ids]}"
    assert set(pred) == set(queries), f"query ids differ: missing {len(set(queries) - set(pred))}, extra {len(set(pred) - set(queries))}"
    for q, r in pred.items():
        assert isinstance(r, list) and len(r) == k, f"{q}: need exactly {k} candidate ids, got {len(r) if isinstance(r, list) else type(r)}"
        assert len(set(r)) == k, f"{q}: candidate ids must be unique"
        assert all(isinstance(c, str) for c in r), f"{q}: candidate ids must be strings"
        bad = [c for c in r if c not in pool]
        assert not bad, f"{q}: ids not in the candidate pool, e.g. {bad[:3]}"
    return len(pred)


def length_stats(toks):
    n = sorted(len(v) for v in toks.values())
    q = lambda f: int(n[min(len(n) - 1, int(len(n) * f))]) if n else 0
    return dict(n=len(n), p10=q(.1), p50=q(.5), p90=q(.9), p99=q(.99), max=int(n[-1]) if n else 0)

In [ ]:
%%writefile test_pcr_data.py
"""Synthetic tests for pcr_data.py (python test_pcr_data.py)."""
import json, tempfile

import numpy as np

import pcr_data as pd_

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


tmp = tempfile.mkdtemp()

q = [dict(id="0000021652", text=["First sentence.", "Second one."], relevant_candidates=["0000590455", "0000000987"]),
     dict(id="0000030721", text=["Another query."], relevant_candidates=["0000004354"])]
json.dump(q, open(f"{tmp}/q.json", "w"))
json.dump({r["id"]: dict(text=r["text"], relevant_candidates=r["relevant_candidates"]) for r in q}, open(f"{tmp}/q2.json", "w"))
json.dump([dict(id=r["id"], text=r["text"]) for r in q], open(f"{tmp}/qt.json", "w"))
pool = [dict(id=f"{i:010d}", text=[f"cand {i}"]) for i in range(20)] + [dict(id="0000590455", text="x y"), dict(id="0000004354", text=["z"])]
json.dump(pool, open(f"{tmp}/p.json", "w"))
json.dump({r["id"]: r["text"] for r in pool}, open(f"{tmp}/p2.json", "w"))
a, b, t = pd_.load_queries(f"{tmp}/q.json"), pd_.load_queries(f"{tmp}/q2.json"), pd_.load_queries(f"{tmp}/qt.json")
check("1 query loaders agree + sentences joined", a == b and a["0000021652"]["text"] == "First sentence. Second one.")
check("1 ids kept as zero-padded strings", "0000021652" in a and a["0000021652"]["relevant"] == ["0000590455", "0000000987"])
check("1 test queries have no labels", all(v["relevant"] is None for v in t.values()))
P, P2 = pd_.load_pool(f"{tmp}/p.json"), pd_.load_pool(f"{tmp}/p2.json")
check("1 pool loaders agree", P == P2 and len(P) == 22)


ids = np.arange(1000, dtype=np.uint16)
c = pd_.chunk_ids(ids, 256, 3, "first")
check("2 first", len(c) == 3 and c[0][0] == 0 and c[1][0] == 256 and all(len(x) == 256 for x in c))
c = pd_.chunk_ids(ids, 256, 2, "head_tail")
check("2 head_tail", len(c) == 2 and c[0][0] == 0 and c[-1][-1] == 999)
c = pd_.chunk_ids(ids, 256, 3, "spread")
check("2 spread covers start and end", len(c) == 3 and c[0][0] == 0 and c[-1][-1] == 999)
check("2 short doc = one chunk", len(pd_.chunk_ids(ids[:100], 256, 4)) == 1 and len(pd_.chunk_ids(ids[:100], 256, 4)[0]) == 100)
check("2 stride overlap", pd_.chunk_ids(ids, 256, 3, "first", stride=128)[1][0] == 128)
check("2 empty doc -> one token", len(pd_.chunk_ids(np.zeros(0, np.uint16), 256, 2)[0]) == 1)


ranked = {"q1": ["a", "b", "c", "d", "e", "f", "g"], "q2": ["x", "y", "z", "u", "v", "w", "s"]}
gold = {"q1": ["a", "c", "h"], "q2": ["s", "t"]}
m = pd_.micro_at_k(ranked, gold, 7)

check("3 micro P/R", abs(m["precision"] - 3 / 14) < 1e-12 and abs(m["recall"] - 3 / 5) < 1e-12)
check("3 micro F1", abs(m["f1"] - 2 * (3 / 14) * (3 / 5) / (3 / 14 + 3 / 5)) < 1e-12)
check("3 only top-7 count", pd_.micro_at_k({"q": list("abcdefgh")}, {"q": ["h"]})["tp"] == 0)


docs = {"d0": np.array([1, 2, 3, 4]), "d1": np.array([5, 6, 7, 8]), "d2": np.array([1, 1, 9, 9])}
bm = pd_.BM25(docs)
s = bm.scores(np.array([9, 1]))
check("4 bm25 ranking", pd_.rank_pool(s, bm.keys)[0] == "d2" and s[1] == 0)
check("4 exclude", pd_.rank_pool(s, bm.keys, exclude=["d2"])[0] == "d0")


good = {qid: [f"{i:010d}" for i in range(7)] for qid in t}
check("5 accepts", pd_.validate_prediction(good, t, P) == 2)
bads = [({k: v[:6] for k, v in good.items()}, "6 ids"), ({k: v[:6] + v[:1] for k, v in good.items()}, "duplicate"),
        ({k: v[:6] + ["9999999999"] for k, v in good.items()}, "not in pool"), (dict(list(good.items())[:1]), "missing query")]
for bad, why in bads:
    try:
        pd_.validate_prediction(bad, t, P); check(f"5 rejects {why}", False)
    except AssertionError:
        check(f"5 rejects {why}", True)

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

In [ ]:
%%writefile pcr_train.py
"""pcr_train.py — Phase-7 prior case retrieval harness."""
import importlib, json, math, os, random, sys, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F

import pcr_data as pdat

CFG = json.load(open(sys.argv[1]))
D, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]
KD = CFG.get("kd") or {}
OUT, NAME = CFG["out_dir"], CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
K = 7


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))


def enc_batch(texts):
    return [e.ids for e in TOK.encode_batch(texts)]


cache = D.get("cache_dir", "pcr_cache")
SPL = {}
for s in ("train", "dev", "test"):
    if not D.get(f"{s}_q"):
        continue
    q, p = pdat.load_queries(D[f"{s}_q"]), pdat.load_pool(D[f"{s}_pool"])
    qt = pdat.tokenize_texts({k: v["text"] for k, v in q.items()}, enc_batch, cache, f"{s}-q")
    pt = pdat.tokenize_texts({k: v["text"] for k, v in p.items()}, enc_batch, cache, f"{s}-pool")
    gold = {k: [c for c in v["relevant"] if c in p and c != k] for k, v in q.items() if v["relevant"] is not None}
    SPL[s] = dict(q=q, pool=p, qt=qt, pt=pt, gold={k: g for k, g in gold.items() if g}, pkeys=list(p))
    miss = sum(len(v["relevant"]) for v in q.values() if v["relevant"]) - sum(len(g) for g in gold.values())
    log(f"{s}: {len(q)} queries | pool {len(p)} | queries with >=1 relevant in pool {len(SPL[s]['gold'])} | "
        f"relevant per query mean {np.mean([len(g) for g in SPL[s]['gold'].values()]) if SPL[s]['gold'] else 0:.2f} | "
        f"relevant ids missing from the pool {miss} | query tokens {pdat.length_stats(qt)} | pool tokens {pdat.length_stats(pt)}")

L_CH, N_CH = int(D["chunk_len"]), int(D["n_chunks"])
MODE, STRIDE = D.get("chunk_mode", "first"), D.get("stride")
TRAIN_CH = int(D.get("train_chunks") or N_CH)
PICK = D.get("train_chunk_pick", "first")
AGG, SIM = D.get("agg", "mean"), D.get("sim", "cos")
assert AGG in ("mean", "max", "first") and SIM in ("cos", "dot"), "agg: mean|max|first, sim: cos|dot"


def chunks_of(ids, train=False):
    ch = pdat.chunk_ids(ids, L_CH, N_CH, MODE, STRIDE)
    if train and len(ch) > TRAIN_CH:
        ch = random.sample(ch, TRAIN_CH) if PICK == "random" else ch[:TRAIN_CH]
    return ch


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "pcr_model"))
model = model_lib.build_pcr_model(dict(MODEL_CFG)).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
if CFG.get("load_trained"):
    sd_ = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]
    cur = dict(model.named_parameters())
    miss = [k for k in sd_ if k not in cur]
    assert not miss, f"load_trained: names not in this model, e.g. {miss[:3]}"
    with torch.no_grad():
        for k, v in sd_.items():
            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))
    log(f"loaded {len(sd_)} trained tensors from {CFG['load_trained']}")
assert n_train > 0 or int(TR.get("epochs", 0)) == 0, "no trainable parameters"
CH_BS = int(TR.get("chunk_batch", 64))


def embed_chunks(chunks):
    groups = {}
    for i, c in enumerate(chunks):
        groups.setdefault(len(c), []).append(i)
    out, idx = [], []
    for L, ii in groups.items():
        for j in range(0, len(ii), CH_BS):
            sub = ii[j:j + CH_BS]
            x = torch.tensor(np.stack([chunks[i] for i in sub]).astype(np.int64), device=DEV)
            with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
                e = model.embed(x)
            assert e.dim() == 2 and e.shape[0] == len(sub), f"embed must return [B, D], got {tuple(e.shape)}"
            out.append(e.float()); idx += sub
    E = torch.cat(out, 0)
    inv = torch.empty(len(idx), dtype=torch.long, device=E.device)
    inv[torch.tensor(idx, device=E.device)] = torch.arange(len(idx), device=E.device)
    return E[inv]


def doc_vectors(id_lists, train=False):
    chs, owner = [], []
    for d, ids in enumerate(id_lists):
        c = chunks_of(ids, train)
        chs += c; owner += [d] * len(c)
    E = embed_chunks(chs)
    owner = torch.tensor(owner, device=E.device)
    n = len(id_lists)
    if AGG == "mean":
        V = torch.zeros(n, E.shape[1], device=E.device).index_add_(0, owner, E)
        V = V / torch.bincount(owner, minlength=n).clamp(min=1)[:, None].float()
    elif AGG == "max":
        V = torch.full((n, E.shape[1]), -1e4, device=E.device).scatter_reduce(0, owner[:, None].expand_as(E), E, "amax")
    else:
        cnt = torch.bincount(owner, minlength=n)
        V = E[torch.cumsum(cnt, 0) - cnt]
    return V


def sim(Q, C):
    if SIM == "cos":
        Q, C = F.normalize(Q, dim=-1), F.normalize(C, dim=-1)
    return Q @ C.T


@torch.no_grad()
def encode_all(tok_map, keys, bs_docs=None):
    model.eval()
    bs_docs = bs_docs or max(1, CH_BS // max(1, N_CH))
    order = sorted(range(len(keys)), key=lambda i: len(tok_map[keys[i]]))
    vecs = torch.zeros(len(keys), 1)
    out = [None] * len(keys)
    for i in range(0, len(order), bs_docs):
        b = order[i:i + bs_docs]
        V = doc_vectors([tok_map[keys[j]] for j in b])
        for j, v in zip(b, V):
            out[j] = v.cpu()
    model.train()
    return torch.stack(out) if out else vecs


@torch.no_grad()
def rank_split(s, depth=100):
    S = SPL[s]
    qk, pk = list(S["q"]), S["pkeys"]
    Qv, Pv = encode_all(S["qt"], qk), encode_all(S["pt"], pk)
    sc = sim(Qv.to(DEV), Pv.to(DEV)).float().cpu().numpy()
    ranked = {q: pdat.rank_pool(sc[i], pk, exclude=[q], k=depth) for i, q in enumerate(qk)}
    return ranked, Qv, Pv


def dev_scores(ranked, gold):
    m = pdat.micro_at_k(ranked, gold, K)
    return dict(f1=m["f1"], p=m["precision"], r=m["recall"], r50=pdat.recall_at(ranked, gold, 50),
                r100=pdat.recall_at(ranked, gold, 100))


# ------------------------------------------------------------------ negatives
TRS = SPL.get("train")
N_NEG = int(TR.get("n_neg", 0))
NEG_SRC = TR.get("neg_source", "random")
HARD_POOL = int(TR.get("hard_pool", 50))
HARD = {}
T_EMB = None
if KD.get("teacher_emb") or NEG_SRC == "teacher":
    path = KD.get("teacher_emb") or TR.get("teacher_emb")
    assert path, "neg_source='teacher' needs kd.teacher_emb (or train.teacher_emb)"
    te = torch.load(path, map_location="cpu", weights_only=False)
    T_EMB = dict(q={k: i for i, k in enumerate(te["q_keys"])}, p={k: i for i, k in enumerate(te["p_keys"])},
                 Q=te["Q"].float(), P=te["P"].float(), sim=te["sim"])
    log(f"teacher vectors: {len(te['q_keys'])} queries, {len(te['p_keys'])} candidates (run {te['run']}, sim {te['sim']})")


def t_sim(Q, C):
    if T_EMB["sim"] == "cos":
        Q, C = F.normalize(Q, dim=-1), F.normalize(C, dim=-1)
    return Q @ C.T


if TRS and int(TR.get("epochs", 0)) > 0 and N_NEG and NEG_SRC in ("bm25", "teacher", "mix"):
    t1 = time.time()
    pk = TRS["pkeys"]
    use_teacher = NEG_SRC == "teacher" or (NEG_SRC == "mix" and T_EMB is not None and TR.get("mix_hard") == "teacher")
    if use_teacher:
        qk = [q for q in TRS["gold"] if q in T_EMB["q"]]
        Qt = T_EMB["Q"][[T_EMB["q"][q] for q in qk]]
        Pt = T_EMB["P"][[T_EMB["p"][c] for c in pk]]
        sc = t_sim(Qt, Pt).numpy()
        for i, q in enumerate(qk):
            HARD[q] = pdat.rank_pool(sc[i], pk, exclude=[q] + TRS["gold"][q], k=HARD_POOL)
    else:
        bm = pdat.BM25({c: TRS["pt"][c] for c in pk}, max_terms=TR.get("bm25_max_terms"))
        for q in TRS["gold"]:
            HARD[q] = pdat.rank_pool(bm.scores(TRS["qt"][q], TR.get("bm25_max_terms")), bm.keys,
                                     exclude=[q] + TRS["gold"][q], k=HARD_POOL)
    log(f"hard negatives ({'teacher' if use_teacher else 'bm25'}): top {HARD_POOL} non-relevant for {len(HARD)} queries "
        f"({time.time() - t1:.0f}s)")


def sample_negs(q, n):
    if not n:
        return []
    bad = set(TRS["gold"][q]) | {q}
    out = []
    if NEG_SRC in ("bm25", "teacher", "mix") and HARD.get(q):
        nh = n if NEG_SRC != "mix" else (n + 1) // 2
        out = random.sample(HARD[q], min(nh, len(HARD[q])))
    tries = 0
    while len(out) < n and tries < 50 * n:
        c = random.choice(TRS["pkeys"]); tries += 1
        if c not in bad and c not in out:
            out.append(c)
    while len(out) < n:
        out.append(random.choice([c for c in TRS["pkeys"] if c not in bad] or TRS["pkeys"]))
    return out


# ------------------------------------------------------------------ loss
TAU = float(TR.get("tau", 0.05))
LOSS = TR.get("loss", "infonce")
IN_BATCH = bool(TR.get("in_batch", True))
ALPHA, KD_TAU = float(KD.get("alpha", 0.0)), float(KD.get("kd_tau", 1.0))


def step_loss(qs):
    pos = [random.choice(TRS["gold"][q]) for q in qs]
    negs = [sample_negs(q, N_NEG) for q in qs]
    cols = pos + [c for ns in negs for c in ns]
    Qv = doc_vectors([TRS["qt"][q] for q in qs], train=True)
    Cv = doc_vectors([TRS["pt"][c] for c in cols], train=True)
    S = sim(Qv, Cv)
    B = len(qs)
    own = torch.zeros_like(S, dtype=torch.bool)
    for i in range(B):
        own[i, i] = True
        own[i, B + i * N_NEG: B + (i + 1) * N_NEG] = True
    allowed = own.clone() if not IN_BATCH else torch.ones_like(own)
    for i, q in enumerate(qs):
        g = set(TRS["gold"][q])
        for j, c in enumerate(cols):
            if j != i and c in g:
                allowed[i, j] = False
    neg_inf = -1e4
    if LOSS == "infonce":
        task = F.cross_entropy((S / TAU).masked_fill(~allowed, neg_inf), torch.arange(B, device=S.device))
    elif LOSS == "pairwise":
        m = float(TR.get("margin", 0.2))
        pos_s = S[torch.arange(B), torch.arange(B)][:, None]
        negm = allowed.clone(); negm[torch.arange(B), torch.arange(B)] = False
        task = (F.relu(m - pos_s + S) * negm).sum() / negm.sum().clamp(min=1)
    else:
        raise ValueError("loss must be infonce | pairwise")
    if ALPHA and T_EMB is not None:
        ok = all(q in T_EMB["q"] for q in qs) and all(c in T_EMB["p"] for c in cols)
        if ok:
            Qt = T_EMB["Q"][[T_EMB["q"][q] for q in qs]].to(S.device)
            Ct = T_EMB["P"][[T_EMB["p"][c] for c in cols]].to(S.device)
            St = t_sim(Qt, Ct)
            ls = F.log_softmax((S / KD_TAU).masked_fill(~allowed, neg_inf), -1)
            pt = F.softmax((St / KD_TAU).masked_fill(~allowed, neg_inf), -1)
            kd = (pt * (torch.log(pt.clamp(min=1e-12)) - ls)).masked_fill(~allowed, 0).sum(-1).mean()
            return (1 - ALPHA) * task + ALPHA * kd, float(task), float(kd)
    return task, float(task), 0.0


# ------------------------------------------------------------------ training
epochs = int(TR.get("epochs", 0))
TQ = list(TRS["gold"]) if TRS else []
if TR.get("train_subset"):
    TQ = random.Random(SEED).sample(TQ, min(len(TQ), int(TR["train_subset"])))
bs, acc = int(TR.get("batch_size", 4)), int(TR.get("grad_accum", 1))
per_epoch = min(len(TQ), int(TR.get("epoch_queries") or len(TQ))) if TQ else 0
steps_per_epoch = math.ceil(per_epoch / (bs * acc)) if per_epoch else 0
total_steps = max(1, steps_per_epoch * epochs)
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR.get("lr", 1e-4))
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0))) if n_train else None
if opt:
    for g in opt.param_groups:
        g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
SELECT = TR.get("select_metric", "dev_f1")
EVAL_EVERY = int(TR.get("eval_every", 1))
log(f"chunks: {N_CH} x {L_CH} ({MODE}, stride {STRIDE or L_CH}) at inference, {TRAIN_CH} ({PICK}) in training | agg {AGG} | "
    f"sim {SIM} | loss {LOSS} tau {TAU} | n_neg {N_NEG} ({NEG_SRC}) | in_batch {IN_BATCH} | KD alpha {ALPHA}")
log(f"train queries/epoch {per_epoch} | steps/epoch {steps_per_epoch} | epochs {epochs} | select by {SELECT}")


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    t = (step - warm) / max(1, total_steps - warm)
    sch = TR.get("schedule", "linear")
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


def snap():
    return {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}


history, best, best_state = [], None, snap()
if "dev" in SPL and SPL["dev"]["gold"] and TR.get("eval_before", True):
    t1 = time.time()
    rk, _, _ = rank_split("dev")
    sc0 = dev_scores(rk, SPL["dev"]["gold"])
    history.append(dict(epoch=0, **{f"dev_{k}": v for k, v in sc0.items()}))
    best = sc0[SELECT[4:]]
    log("epoch 0 (no training) | " + " | ".join(f"dev_{k} {v:.4f}" for k, v in sc0.items()) + f" | eval {time.time()-t1:.0f}s")
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
order = []
model.train()
stop = False
for ep in range(1, epochs + 1):
    if len(order) < per_epoch:
        rest = TQ[:]; random.shuffle(rest); order += rest
    ep_q, order = order[:per_epoch], order[per_epoch:]
    run, run_t, run_k, nb = 0.0, 0.0, 0.0, 0
    for i in range(0, len(ep_q), bs):
        loss, lt, lk = step_loss(ep_q[i:i + bs])
        scaler.scale(loss / acc).backward()
        nb += 1; run += loss.item(); run_t += lt; run_k += lk
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            stop = True; break
    rec = dict(epoch=ep, train_loss=run / max(1, nb), task_loss=run_t / max(1, nb), minutes=(time.time() - t0) / 60)
    if ALPHA:
        rec["kd_loss"] = run_k / max(1, nb)
    if "dev" in SPL and SPL["dev"]["gold"] and (ep % EVAL_EVERY == 0 or ep == epochs or stop):
        rk, _, _ = rank_split("dev")
        rec.update({f"dev_{k}": v for k, v in dev_scores(rk, SPL["dev"]["gold"]).items()})
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    if SELECT not in rec and not ("dev" in SPL and SPL["dev"]["gold"]):
        best_state = snap()
    if SELECT in rec and (best is None or rec[SELECT] >= best):
        best = rec[SELECT]; best_state = snap()
        log(f"  best so far (epoch {ep})")
    if stop:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0
cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG), os.path.join(OUT, f"{NAME}_trained_params.pt"))
result = dict(run=NAME, data={k: v for k, v in D.items() if not k.endswith(("_q", "_pool"))}, train=TR, kd=KD or None,
              params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb, history=history)

# ------------------------------------------------------------------ final dev evaluation (best weights)
if "dev" in SPL and SPL["dev"]["gold"]:
    t1 = time.time()
    rk, _, _ = rank_split("dev")
    sc = dev_scores(rk, SPL["dev"]["gold"])
    result["dev"] = sc; result["dev_eval_seconds"] = time.time() - t1
    log("DEV (best weights): " + " | ".join(f"{k} {v:.4f}" for k, v in sc.items()) + f" | {time.time()-t1:.0f}s")
    json.dump({q: dict(ranked=rk[q][:50], gold=SPL["dev"]["gold"][q]) for q in SPL["dev"]["gold"]},
              open(os.path.join(OUT, f"{NAME}_dev_rankings.json"), "w"))

# ------------------------------------------------------------------ teacher vectors for KD (train split)
if CFG.get("dump_teacher") and TRS:
    t1 = time.time()
    qk, pk = list(TRS["q"]), TRS["pkeys"]
    torch.save(dict(run=NAME, sim=SIM, q_keys=qk, p_keys=pk, Q=encode_all(TRS["qt"], qk).half(),
                    P=encode_all(TRS["pt"], pk).half()), os.path.join(OUT, f"{NAME}_teacher_emb.pt"))
    log(f"teacher vectors written for {len(qk)} train queries + {len(pk)} train candidates ({(time.time()-t1)/60:.1f} min)")

# ------------------------------------------------------------------ test
if "test" in SPL:
    t1 = time.time()
    rk, _, _ = rank_split("test", depth=K)
    pred = {q: rk[q][:K] for q in SPL["test"]["q"]}
    n = pdat.validate_prediction(pred, SPL["test"]["q"], SPL["test"]["pool"], K)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w"), indent=1)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    result.update(test_queries=n, test_seconds=time.time() - t1)
    log(f"test: {n} queries x {K} ids | {(time.time()-t1)/60:.1f} min | wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1, default=str)
log(f"done: {NAME} | best {SELECT} {best if best is not None else float('nan'):.4f} | train {train_min:.1f} min | "
    f"peak {peak_gb:.1f} GB | trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

In [ ]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

In [ ]:
%%writefile pcr_dummy.py
"""Smoke-test model only (NOT a submission model): tiny bag-of-embeddings encoder that satisfies the pcr_train contract."""
import torch
import torch.nn as nn


class Dummy(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = int(cfg.get("d", 64))
        self.emb = nn.Embedding(int(cfg.get("vocab", 32000)), d)
        self.proj = nn.Linear(d, d)
        self.keep = int(cfg.get("keep_dims", d))

    def embed(self, ids):
        return self.proj(self.emb(ids).mean(1))[:, :self.keep]

    def param_groups(self, base_lr):
        return [{"params": [p for p in self.parameters() if p.requires_grad], "lr": base_lr}]


def build_pcr_model(model_cfg):
    return Dummy(model_cfg)

In [ ]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

In [ ]:
# 3 — data check: layouts, sizes, relevance labels, token lengths, BM25 reference (lexical only, NOT a submission); builds token caches
!python test_pcr_data.py | tail -1
import pcr_data as pdat, importlib; importlib.reload(pdat)
import numpy as np
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
V = _tok.get_vocab_size()
enc_batch = lambda texts: [e.ids for e in _tok.encode_batch(texts)]
for k, p in P7.items():
    raw = json.load(open(p))
    ex = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{k:10s} {type(raw).__name__} of {len(raw)} | example: {str(ex)[:220]} ...")
DATA = {}
t0 = time.time()
for s in ("train", "dev", "test"):
    q, pool = pdat.load_queries(P7[f"{s}_q"]), pdat.load_pool(P7[f"{s}_pool"])
    qt = pdat.tokenize_texts({k: v["text"] for k, v in q.items()}, enc_batch, "pcr_cache", f"{s}-q")
    pt = pdat.tokenize_texts({k: v["text"] for k, v in pool.items()}, enc_batch, "pcr_cache", f"{s}-pool")
    DATA[s] = dict(q=q, pool=pool, qt=qt, pt=pt)
print(f"tokenized in {time.time()-t0:.0f}s\n")
for s, d in DATA.items():
    q, pool = d["q"], d["pool"]
    rel = [v["relevant"] for v in q.values() if v["relevant"] is not None]
    inpool = [sum(c in pool for c in r) for r in rel]
    print(f"{s}: {len(q)} queries | pool {len(pool)} | labelled {len(rel)} | queries whose own id is in the pool {sum(k in pool for k in q)}")
    if rel:
        n = [len(r) for r in rel]
        print(f"   relevant per query: mean {np.mean(n):.2f} | p10 {np.percentile(n,10):.0f} p50 {np.percentile(n,50):.0f} p90 {np.percentile(n,90):.0f} max {max(n)} | "
              f"queries with >7 relevant {np.mean([x > 7 for x in n]):.0%} | relevant ids found in pool {sum(inpool)}/{sum(n)}")
        print(f"   ceiling of Micro-F1@7 if all 7 slots were perfect: {pdat.micro_at_k({k: (v['relevant'] + ['_']*7)[:7] for k, v in q.items() if v['relevant']}, {k: v['relevant'] for k, v in q.items() if v['relevant']})['f1']:.3f}")
    print(f"   query tokens {pdat.length_stats(d['qt'])}\n   pool tokens  {pdat.length_stats(d['pt'])}")
print("pool overlap train/dev:", len(set(DATA['train']['pool']) & set(DATA['dev']['pool'])), "| train/test:", len(set(DATA['train']['pool']) & set(DATA['test']['pool'])))
for s in ("train", "dev"):
    d = DATA[s]; bm = pdat.BM25(d["pt"])
    gold = {k: [c for c in v["relevant"] if c in d["pool"] and c != k] for k, v in d["q"].items() if v["relevant"]}
    rk = {k: pdat.rank_pool(bm.scores(d["qt"][k]), bm.keys, exclude=[k], k=100) for k in gold}
    m = pdat.micro_at_k(rk, gold)
    print(f"BM25 reference on {s}: Micro-F1@7 {m['f1']:.4f} | P@7 {m['precision']:.4f} | R@7 {m['recall']:.4f} | "
          f"R@50 {pdat.recall_at(rk, gold, 50):.4f} | R@100 {pdat.recall_at(rk, gold, 100):.4f}")
k0 = next(iter(DATA["train"]["q"])); print("\nexample query text:", DATA["train"]["q"][k0]["text"][:500])

In [ ]:
# 4 — Phase-1 models -> phase1_registry.json (same as Phase 2)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach phase1-models)"); continue
    shutil.copy(near(ck[0], "model.py"), f"p1_{name}.py")
    cfgj = json.load(open(near(ck[0], "run_config.json")))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

In [ ]:
# 5 — experiment config helper + runners
TODO = "TODO"
R = "/kaggle/working/p7runs"
def make_cfg(run_name, model, train, data=None, kd=None, module="pcr_model", dump_teacher=False, splits=("train", "dev", "test")):
    d = dict(tokenizer_dir="tokenizer", cache_dir="pcr_cache",
             chunk_len=TODO,
             n_chunks=TODO,
             chunk_mode=TODO,
             stride=None,
             train_chunks=None,
             train_chunk_pick="first",
             agg=TODO,
             sim=TODO)
    for s in ("train", "dev", "test"):
        d[f"{s}_q"] = P7[f"{s}_q"] if s in splits else None
        d[f"{s}_pool"] = P7[f"{s}_pool"] if s in splits else None
    d.update(data or {})


    cfg = dict(run_name=run_name, out_dir=f"{R}/{run_name}", model_module=module, data=d, model=model, train=train,
               kd=kd, dump_teacher=dump_teacher)
    todo = [k for k, v in {**d, **train, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_on_gpu(cp, gpu):
    lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    subprocess.Popen([sys.executable, "pcr_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    print("started", cp, "on GPU", gpu)

def tail_log(name, n=10):
    print("\n".join(open(f"{R}/{name}/train.log").read().splitlines()[-n:]))

def teacher_from(run, new):
    c = json.load(open(f"{R}/{run}/config.json"))
    p = make_cfg(new, model=c["model"], train=dict(c["train"], epochs=0, eval_before=False), data=c["data"],
                 module=c["model_module"], dump_teacher=True, splits=("train",))
    cfg = json.load(open(p)); cfg["load_trained"] = f"{R}/{run}/{run}_trained_params.pt"
    for s in ("dev", "test"):
        cfg["data"][f"{s}_q"] = cfg["data"][f"{s}_pool"] = None
    json.dump(cfg, open(p, "w"), indent=1); return p

def _cfg_for(job):
    if job.get("teacher_of"):
        return teacher_from(job["teacher_of"], job["name"])
    kd = None
    if job.get("kd"):
        k = dict(job["kd"]); t = k.pop("teacher")
        kd = dict(teacher_emb=f"{R}/{t}_teacher/{t}_teacher_teacher_emb.pt", **k)
    return make_cfg(job["name"], job["model"], job["train"], job.get("data"), kd=kd, module=job.get("module", "pcr_model"))

def run_queues(queues, poll=20, deadline_h=11.25, reserve_min=60):
    import signal
    start = globals().get("NB_START", time.time())
    left_min = lambda: deadline_h * 60 - (time.time() - start) / 60
    qs = {}
    for g, jobs in queues.items():
        lst = []
        for j in jobs:
            if j.get("kd"):
                t = j["kd"]["teacher"]
                lst.append(dict(name=f"{t}_teacher", teacher_of=t))
            lst.append(j)
        qs[int(g)] = lst
    cur, t_start = {}, time.time()
    while any(qs.values()) or cur:
        for g in list(qs):
            if g not in cur and qs[g]:
                j = qs[g].pop(0)
                res = j.get("reserve_min", reserve_min)
                if left_min() - res < 10:
                    print(f"SKIPPED {j['name']} on GPU {g}: only {left_min():.0f} min left before the {deadline_h} h guard", flush=True)
                    continue
                try:
                    cp = _cfg_for(j)
                except Exception as e:
                    print(f"SKIPPED {j['name']} on GPU {g}: {type(e).__name__}: {e}", flush=True); continue
                c = json.load(open(cp)); own = float(c["train"].get("max_minutes", 1e9)); cap = left_min() - res
                if cap < own:
                    c["train"]["max_minutes"] = cap; json.dump(c, open(cp, "w"), indent=1)
                    print(f"  {j['name']}: max_minutes {own:.0f} -> {cap:.0f} (12 h guard)", flush=True)
                lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
                p = subprocess.Popen([sys.executable, "pcr_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env,
                                     start_new_session=True)
                cur[g] = (p, j["name"], lf, time.time())
                print(f"[{(time.time()-t_start)/60:6.1f} min] started {j['name']} on GPU {g}", flush=True)
        time.sleep(poll)
        if left_min() < -0.35 * 60:
            for g, (p, name, lf, t0) in cur.items():
                if p.poll() is None:
                    print(f"STOPPING {name} on GPU {g}: past the {deadline_h} h guard", flush=True)
                    try: os.killpg(p.pid, signal.SIGKILL)
                    except Exception: p.kill()
            for g in qs: qs[g] = []
        for g, (p, name, lf, t0) in list(cur.items()):
            if p.poll() is not None:
                lf.close(); del cur[g]
                tail = open(f"{R}/{name}/train.log").read().splitlines()[-14:]
                print(f"\n==== [{(time.time()-t_start)/60:6.1f} min] {name} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n"
                      + "\n".join(tail) + "\n", flush=True)


def summary(names):
    for n in names:
        z, res = f"{R}/{n}/{n}_submission.zip", f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res)); v = r.get("dev", {})
            print(f"OK  {n:26s} dev F1@7 {v.get('f1', float('nan')):.4f} | P@7 {v.get('p', float('nan')):.4f} | R@7 {v.get('r', float('nan')):.4f} | "
                  f"R@50 {v.get('r50', float('nan')):.4f} | trainable {r['params_trainable']/1e6:.2f}M of {r['params_total']/1e6:.1f}M | "
                  f"{r['train_minutes']:.1f} min | peak {r['peak_mem_gb']:.1f} GB -> {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")

In [ ]:
# 6 — smoke test: dummy model; random / BM25 / teacher negatives, InfoNCE + pairwise, mean / max / first agg, KD, zip
SM_D = dict(chunk_len=64, n_chunks=2, chunk_mode="first", agg="mean", sim="cos")
SM_T = dict(epochs=2, epoch_queries=48, train_subset=48, lr=1e-3, batch_size=4, n_neg=2, neg_source="random", in_batch=True,
            loss="infonce", tau=0.05, max_minutes=4, select_metric="dev_f1")
a = make_cfg("smoke_a", dict(vocab=V), SM_T, SM_D, module="pcr_dummy")
b = make_cfg("smoke_b", dict(vocab=V), dict(SM_T, neg_source="bm25", hard_pool=20, loss="pairwise", margin=0.2, in_batch=False,
             select_metric="dev_r50"), dict(SM_D, chunk_mode="spread", agg="max", sim="dot", train_chunks=1, train_chunk_pick="random"),
             module="pcr_dummy")
run_on_gpu(a, 0); run_on_gpu(b, 1)
while not all(os.path.exists(f"{R}/{n}/{n}_result.json") or "Traceback" in open(f"{R}/{n}/train.log").read() for n in ("smoke_a", "smoke_b")):
    time.sleep(10)
run_queues({0: [dict(name="smoke_kd", model=dict(vocab=V, keep_dims=32), train=dict(SM_T, neg_source="teacher", hard_pool=20),
                     data=dict(SM_D, agg="first"), module="pcr_dummy", kd=dict(teacher="smoke_a", alpha=0.5, kd_tau=1.0))]})
for n in ("smoke_a", "smoke_b", "smoke_kd"): tail_log(n, 5)
summary(["smoke_a", "smoke_b", "smoke_kd"])
assert all(os.path.exists(f"{R}/{n}/{n}_submission.zip") for n in ["smoke_a", "smoke_b", "smoke_kd"]), "SMOKE TEST FAILED"
import zipfile; zf = zipfile.ZipFile(f"{R}/smoke_kd/smoke_kd_submission.zip"); pj = json.loads(zf.read("prediction.json"))
print("zip contains:", zf.namelist(), "| queries:", len(pj), "| first entry:", next(iter(pj.items())))
print("SMOKE TEST PASSED (pipeline only: the dummy model's rankings mean nothing)")

In [ ]:
%%writefile sum_model.py
"""sum_model.py — Phase-6 model."""
import math
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from phase1_loader import load_phase1

BLOCKS = {"prefixlm": "blocks", "encdec": "encoder", "arch4": "blocks", "biscale": "blocks"}
FINAL_NORM = {"prefixlm": "ln_f", "encdec": "enc_ln", "arch4": "ln_f", "biscale": "norm"}
DEFAULT_TARGETS = {
    "prefixlm": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "encdec": ["attn.qkv", "attn.proj", "cross.q", "cross.kv", "cross.proj", "mlp.fc1", "mlp.fc2"],
    "arch4": ["msr.proj", "msr.out", "mlp.fc1", "mlp.fc2"],
    "biscale": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "ffn.gate", "ffn.up", "ffn.down"],
}
ADAPTER_SITES = {
    "prefixlm": ("attn", "mlp"), "encdec": ("attn", "mlp"),
    "arch4": ("msr", "mlp"), "biscale": ("attn", "ffn"),
}


def _first_attr(obj, names):
    for n in names:
        if getattr(obj, n, None) is not None:
            return getattr(obj, n)
    raise AttributeError(f"{type(obj).__name__} has none of {names}")


def _out_splits(backbone_name, path, lin, d_model):
    if path.endswith("attn.qkv"):
        return [lin.out_features // 3] * 3
    if path.endswith("cross.kv"):
        return [lin.out_features // 2] * 2
    if backbone_name == "arch4" and path.endswith("msr.proj"):
        vd = (lin.out_features - 2 * d_model) // 2
        return [d_model, d_model, vd, vd]
    return [lin.out_features]


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, splits, variant):
        super().__init__()
        self.base, self.splits, self.variant = base, list(splits), variant
        self.scale = alpha / r
        self.A = nn.ParameterList()
        self.B = nn.ParameterList()
        for d in self.splits:
            if variant == "plus":
                A = torch.empty(r, d)
                nn.init.kaiming_uniform_(A, a=math.sqrt(5))
                B = torch.zeros(d, r)
            elif variant == "pi":
                A = torch.zeros(r, d)
                A[0, 0] = 1.0
                B = torch.randn(d, r) / math.sqrt(r)
                B[:, 0] = 1.0
            else:
                raise ValueError(f"lorma_variant must be 'plus' or 'pi', got {variant!r}")
            self.A.append(nn.Parameter(A))
            self.B.append(nn.Parameter(B))

    def forward(self, x):
        y = F.linear(x, self.base.weight)
        outs = []
        for i, part in enumerate(y.split(self.splits, dim=-1)):
            A, B = self.A[i], self.B[i]
            if self.variant == "plus":
                outs.append(part + F.linear(F.linear(part, A.to(part.dtype)), B.to(part.dtype)) * self.scale)
            else:
                M = B @ A
                d = M.shape[0]
                idx = torch.arange(d, device=M.device)
                R = torch.gather(M, 1, (idx[None, :] - idx[:, None]) % d)
                outs.append(F.linear(part, R.to(part.dtype)))
        y = torch.cat(outs, dim=-1) if len(outs) > 1 else outs[0]
        if self.base.bias is not None:
            y = y + self.base.bias.to(y.dtype)
        return y


class Adapter(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.adapter = Adapter(d, m)

    def _adapt(self, out):
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)

    def forward(self, *args, **kwargs):
        return self._adapt(self.inner(*args, **kwargs))

    def step(self, *args, **kwargs):
        return self._adapt(self.inner.step(*args, **kwargs))

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["inner"], name)


class AttnContext:
    def __init__(self):
        self.key_mask = None
        self.capture = False


class KVStore:
    def __init__(self, n_layers, max_len):
        self.k, self.v = [None] * n_layers, [None] * n_layers
        self.max_len, self.pos = max_len, 0


class PrefixLMAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, mask, kv=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        if o.rope:
            q, k = self.rope_fn(q, cos.to(q.dtype), sin.to(q.dtype)), self.rope_fn(k, cos.to(q.dtype), sin.to(q.dtype))
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        if kv is not None:
            store, li = kv
            if store.k[li] is None:
                shape = (B, o.h, store.max_len, o.hd)
                store.k[li] = torch.empty(shape, device=k.device, dtype=k.dtype)
                store.v[li] = torch.empty(shape, device=v.device, dtype=v.dtype)
            store.k[li][:, :, start:start + T] = k
            store.v[li][:, :, start:start + T] = v
            k, v = store.k[li][:, :, :start + T], store.v[li][:, :, :start + T]
        if self.pk is not None and not self.ctx.capture:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            if mask is not None:
                mask = torch.cat([mask.new_ones(mask.shape[0], 1, T, n), mask.expand(-1, -1, T, -1)], dim=-1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p_drop if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class EncoderAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, causal=False, cache=None, li=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope_fn(q, cos, sin), self.rope_fn(k, cos, sin)
        km = self.ctx.key_mask
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        elif self.pk is not None:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            if km is not None:
                km = torch.cat([km.new_ones(B, n), km], dim=1)
        mask = None if km is None else km[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class SumModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.name = cfg.get("phase1", "prefixlm")
        if self.name not in BLOCKS:
            raise ValueError(f"phase1 must be one of {list(BLOCKS)}, got {self.name!r}")
        self.method = cfg.get("method", "full")
        backbone, self.info = load_phase1(self.name)
        self.backbone = backbone
        self.p1 = sys.modules[type(backbone).__module__]
        self.ctx = AttnContext()
        self.use_decoder = self.name == "encdec"
        self.grad_ckpt = bool(cfg.get("grad_ckpt", False))

        self.emb_name = next(n for n in ("tok_emb", "embed_tokens", "embed", "wte")
                             if isinstance(getattr(backbone, n, None), nn.Embedding))
        self.d = self.emb.embedding_dim
        self.vocab = self.emb.num_embeddings
        self.head_name = next((n for n, mod in backbone.named_children()
                               if isinstance(mod, nn.Linear) and mod.out_features == self.vocab), None)
        self._make_student()
        if cfg.get("backbone_dropout") is not None:
            for mod in backbone.modules():
                if isinstance(mod, nn.Dropout):
                    mod.p = float(cfg["backbone_dropout"])
        if self.name == "encdec":
            self._wrap_encoder_attention()
            self.backbone.grad_ckpt = self.grad_ckpt
        if self.name == "prefixlm":
            self._wrap_prefixlm_attention()

        self.n_prompt = 0
        self.prompt = None
        self._apply_method()

    @property
    def emb(self):
        return getattr(self.backbone, self.emb_name)

    def lm_head(self, h):
        if self.head_name is not None:
            return getattr(self.backbone, self.head_name)(h)
        return F.linear(h, self.emb.weight)

    def _blocks(self):
        return getattr(self.backbone, BLOCKS[self.name])

    def _stacks(self):
        s = [self._blocks()]
        if self.use_decoder:
            s.append(self.backbone.decoder)
        return s

    def _make_student(self):
        k = self.cfg.get("keep_layers")
        kd = self.cfg.get("keep_dec_layers")
        mode = self.cfg.get("keep_mode", "first")

        def pick(n, k):
            if mode == "spread":
                return sorted({round(i * (n - 1) / max(1, k - 1)) for i in range(k)})
            return list(range(k))

        if k:
            blocks = self._blocks()
            idx = pick(len(blocks), int(k))
            setattr(self.backbone, BLOCKS[self.name], nn.ModuleList([blocks[i] for i in idx]))
            self.kept_layers = idx
        if kd and self.name == "encdec":
            dec = self.backbone.decoder
            idx = pick(len(dec), int(kd))
            self.backbone.decoder = nn.ModuleList([dec[i] for i in idx])
            self.kept_dec_layers = idx

    def _wrap_encoder_attention(self):
        for blk in self.backbone.encoder:
            blk.attn = EncoderAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _wrap_prefixlm_attention(self):
        for blk in self.backbone.blocks:
            blk.attn = PrefixLMAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _apply_method(self):
        cfg, m = self.cfg, self.method
        bb = self.backbone
        if m == "full":
            for p in bb.parameters():
                p.requires_grad = True
            return
        for p in bb.parameters():
            p.requires_grad = False
        if m == "partial":
            n_top = int(cfg.get("unfreeze_layers", 4))
            for stack in self._stacks():
                for blk in list(stack)[-n_top:]:
                    for p in blk.parameters():
                        p.requires_grad = True
            for p in getattr(bb, FINAL_NORM[self.name]).parameters():
                p.requires_grad = True
            if self.use_decoder:
                for p in bb.dec_ln.parameters():
                    p.requires_grad = True
            if cfg.get("unfreeze_embeddings", False):
                self.emb.weight.requires_grad = True
        elif m in ("lora", "lorma"):
            self._wrap_linears(m)
        elif m == "adapter":
            self._add_adapters()
        elif m == "prompt":
            self._add_prompt()
        elif m == "prefix":
            self._add_prefix()
        else:
            raise ValueError(f"method must be lora | lorma | adapter | prompt | prefix (or full | partial), got {m!r}")
        if cfg.get("train_norms", False):
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, nn.GroupNorm)) or type(mod).__name__.lower().endswith("rmsnorm"):
                    for p in mod.parameters(recurse=False):
                        p.requires_grad = True

    def _wrap_linears(self, kind):
        cfg = self.cfg
        targets = cfg.get("targets") or DEFAULT_TARGETS[self.name]
        r = int(cfg.get("rank", 8))
        alpha = float(cfg.get("alpha", 2 * r))
        stack_names = [BLOCKS[self.name]] + (["decoder"] if self.use_decoder else [])
        replaced = []
        for sname in stack_names:
            for bi, blk in enumerate(getattr(self.backbone, sname)):
                for path, mod in list(blk.named_modules()):
                    clean = path.replace("orig.", "").replace("inner.", "")
                    if not isinstance(mod, nn.Linear) or not any(clean == t or clean.endswith("." + t)
                                                                 for t in targets):
                        continue
                    parent_path, _, leaf = path.rpartition(".")
                    parent = blk.get_submodule(parent_path) if parent_path else blk
                    if kind == "lora":
                        new = LoRALinear(mod, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                    else:
                        splits = _out_splits(self.name, clean, mod, self.d) if cfg.get("lorma_split", True) \
                            else [mod.out_features]
                        new = LoRMALinear(mod, r, alpha, splits, cfg.get("lorma_variant", "plus"))
                    setattr(parent, leaf, new)
                    replaced.append(f"{sname}.{bi}.{path}")
        if not replaced:
            raise ValueError(f"no linear matched targets {targets}")
        self.adapted = replaced

    def _add_adapters(self):
        m = int(self.cfg.get("adapter_size", 64))
        style = self.cfg.get("adapter_style", "houlsby")
        attn_site, mlp_site = ADAPTER_SITES[self.name]
        sites = [attn_site, mlp_site] if style == "houlsby" else [mlp_site]
        for stack in self._stacks():
            for blk in stack:
                for s in sites:
                    setattr(blk, s, WithAdapter(getattr(blk, s), self.d, m))

    def _add_prompt(self):
        n = int(self.cfg.get("n_prompt", 20))
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.vocab, (n,), generator=g)
        self.prompt = nn.Parameter(self.emb.weight.detach()[ids].clone().float())
        self.n_prompt = n

    def _add_prefix(self):
        if self.name not in ("prefixlm", "encdec"):
            raise ValueError("prefix tuning is implemented for the attention models prefixlm and encdec only")
        n = int(self.cfg.get("n_prefix", 16))
        for blk in self._blocks():
            blk.attn.add_prefix(n)
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.vocab, (1, n), generator=g)
        dev = self.emb.weight.device
        self.ctx.capture = True
        was = self.training
        self.eval()
        with torch.no_grad():
            x = self._embed(ids.to(dev))
            if self.name == "encdec":
                self._encode(x)
            else:
                self._states_prefixlm(x, n)
        self.ctx.capture = False
        self.train(was)

    def backbone_max_len(self):
        bb = self.backbone
        for n in ("max_seq_len", "max_len"):
            if isinstance(getattr(bb, n, None), int):
                return min(getattr(bb, n), 1024)
        c = getattr(bb, "c", None) or getattr(bb, "cfg", None) or {}
        if isinstance(c, dict) and c.get("max_seq_len"):
            return int(c["max_seq_len"])
        return 1024

    def _run(self, blk, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(blk, *args, use_reentrant=False)
        return blk(*args)

    def _embed(self, ids):
        x = self.emb(ids)
        if self.prompt is not None:
            x = torch.cat([self.prompt.to(x.dtype)[None].expand(ids.shape[0], -1, -1), x], dim=1)
        return x

    def _check_len(self, n, what):
        lim = self.backbone_max_len()
        if n > lim:
            raise ValueError(f"{what} = {n} tokens exceeds the Phase-1 context {lim}")

    def _states_prefixlm(self, x, prefix_len):
        bb = self.backbone
        T = x.shape[1]
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T, device=x.device))
        x = bb.drop(x)
        cos = bb.rope_cos[:T] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T] if bb.rope_sin is not None else None
        plen = torch.full((x.shape[0],), prefix_len, dtype=torch.long, device=x.device)
        mask = self.p1.prefix_lm_mask(T, plen, x.device)
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _states_arch4(self, x):
        bb = self.backbone
        T = x.shape[1]
        t = bb.tables(x.device)
        x = bb.drop(x)
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _states_biscale(self, x):
        bb = self.backbone
        drop = getattr(bb, "drop", None)
        if drop is not None:
            x = drop(x)
        for blk in bb.blocks:
            x = self._run(blk, x)
        return _first_attr(bb, ["norm", "ln_f", "final_norm"])(x)

    def _encode(self, x):
        bb = self.backbone
        T = x.shape[1]
        x = bb.drop(x)
        cos, sin = bb.rope_cos[:T], bb.rope_sin[:T]
        for blk in bb.encoder:
            x = self._run(blk, x, cos, sin)
        return bb.enc_ln(x)

    def logits(self, src, tgt_in):
        B, S = src.shape
        T = tgt_in.shape[1]
        if self.name == "encdec":
            self._check_len(self.n_prompt + S, "prompt + source")
            self._check_len(T, "summary")
            mem = self._encode(self._embed(src))
            return self.backbone.decode(tgt_in, mem)
        P = self.n_prompt
        self._check_len(P + S + T, "prompt + source + summary")
        ids = torch.cat([src, tgt_in], dim=1)
        if self.name == "biscale" and self.prompt is None:
            return self.backbone(ids, torch.full((B,), S, dtype=torch.long, device=ids.device))[:, S:S + T]
        x = self._embed(ids)
        if self.name == "prefixlm":
            h = self._states_prefixlm(x, P + S)
        elif self.name == "arch4":
            h = self._states_arch4(x)
        else:
            h = self._states_biscale(x)
        return self.lm_head(h[:, P + S:P + S + T])

    def forward(self, src, tgt_in):
        return self.logits(src, tgt_in)

    @torch.no_grad()
    def prefill(self, src, tgt_in):
        if self.name == "biscale":
            raise NotImplementedError("Bi-Scale has no cached decoding; the harness recomputes logits() each step")
        bb = self.backbone
        if self.name == "encdec":
            self._check_len(self.n_prompt + src.shape[1], "prompt + source")
            mem = self._encode(self._embed(src))
            cache = self.p1.EncDecCache(len(bb.decoder), bb.max_len)
            for li, blk in enumerate(bb.decoder):
                cache.ck[li], cache.cv[li] = blk.cross.memory_kv(mem)
            return bb._decode_cached(tgt_in, cache), cache
        P, S = self.n_prompt, src.shape[1]
        ids = torch.cat([src, tgt_in], dim=1)
        x = self._embed(ids)
        T0 = x.shape[1]
        self._check_len(T0, "prompt + source + start token")
        if self.name == "arch4":
            t = bb.tables(x.device)
            cos, sin, mask = t["cos"][:T0], t["sin"][:T0], t["mask"][:, :T0, :T0]
            w = t["powers"][:, :T0].flip(-1)
            states = []
            for blk in bb.blocks:
                x, st = blk.forward_state(x, cos, sin, mask, w)
                states.append(st)
            return self.lm_head(bb.ln_f(x[:, -1])), {"states": states, "pos": T0}
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T0, device=x.device))
        cos = bb.rope_cos[:T0] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T0] if bb.rope_sin is not None else None
        plen = torch.full((x.shape[0],), P + S, dtype=torch.long, device=x.device)
        mask = self.p1.prefix_lm_mask(T0, plen, x.device)
        store = KVStore(len(bb.blocks), self.backbone_max_len())
        for li, blk in enumerate(bb.blocks):
            x = blk(x, cos, sin, mask, (store, li), 0)
        store.pos = T0
        return self.lm_head(bb.ln_f(x[:, -1])), store

    @torch.no_grad()
    def decode_step(self, next_ids, cache):
        bb = self.backbone
        if self.name == "encdec":
            return bb._decode_cached(next_ids, cache), cache
        if self.name == "arch4":
            return bb.decode_step(next_ids, cache)
        n = cache.pos
        self._check_len(n + 1, "generated sequence")
        x = self.emb(next_ids)
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(n, n + 1, device=x.device))
        cos = bb.rope_cos[n:n + 1] if bb.rope_cos is not None else None
        sin = bb.rope_sin[n:n + 1] if bb.rope_sin is not None else None
        for li, blk in enumerate(bb.blocks):
            x = blk(x, cos, sin, None, (cache, li), n)
        cache.pos = n + 1
        return self.lm_head(bb.ln_f(x[:, -1])), cache

    def param_groups(self, base_lr):
        new_mult = float(self.cfg.get("peft_lr_mult", 1.0))
        bb_mult = float(self.cfg.get("backbone_lr_mult", 1.0))
        wd = self.cfg.get("weight_decay")
        groups = {"new": [], "new_nd": [], "bb": [], "bb_nd": []}
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            nd = p.dim() < 2 or n == "prompt" or n.endswith((".pk", ".pv"))
            new = (n == "prompt" or n.endswith((".pk", ".pv", ".A", ".B")) or ".A." in n or ".B." in n
                   or ".adapter." in n)
            groups[("new" if new else "bb") + ("_nd" if nd else "")].append(p)
        mult = {"new": new_mult, "bb": bb_mult}
        out = []
        for k, ps in groups.items():
            if not ps:
                continue
            g = {"params": ps, "lr": base_lr * mult[k.split("_")[0]]}
            if k.endswith("_nd"):
                g["weight_decay"] = 0.0
            elif wd is not None:
                g["weight_decay"] = float(wd)
            out.append(g)
        return out


def build_sum_model(model_cfg: dict) -> nn.Module:
    return SumModel(dict(model_cfg))


@torch.no_grad()
def check_init_equivalence(model_cfg: dict, src_len: int = 256, tgt_len: int = 32, batch: int = 2, seed: int = 0):
    adapted = SumModel(dict(model_cfg)).eval()
    plain = SumModel(dict(model_cfg, method="full")).eval()
    dev = adapted.emb.weight.device
    plain.to(dev)
    g = torch.Generator().manual_seed(seed)
    src = torch.randint(1000, adapted.vocab, (batch, src_len), generator=g).to(dev)
    tgt = torch.randint(1000, adapted.vocab, (batch, tgt_len), generator=g).to(dev)
    a, p = adapted.logits(src, tgt).float(), plain.logits(src, tgt).float()
    diff = (a - p).abs().max().item()
    n_all = sum(q.numel() for q in adapted.parameters())
    n_tr = sum(q.numel() for q in adapted.parameters() if q.requires_grad)
    print(f"{model_cfg.get('phase1')}/{model_cfg.get('method')}: max |logits(adapted) - logits(pretrained)| = "
          f"{diff:.2e} (expect ~0 except prompt/prefix) | trainable {n_tr/1e6:.3f}M of {n_all/1e6:.2f}M "
          f"({100*n_tr/n_all:.2f}%)")
    if plain.name in ("prefixlm", "arch4", "biscale") and not model_cfg.get("keep_layers"):
        ids = torch.cat([src, tgt], 1)
        ref = plain.backbone(ids, torch.full((batch,), src_len, dtype=torch.long, device=dev))[:, src_len:].float()
        print(f"  plain logits() vs Phase-1 forward: max diff {(p - ref).abs().max().item():.2e} (expect ~0)")
    if hasattr(adapted, "prefill") and adapted.name != "biscale":
        last, cache = adapted.prefill(src, tgt[:, :1])
        errs = [(last.float() - a[:, 0]).abs().max().item()]
        for t in range(1, tgt_len):
            last, cache = adapted.decode_step(tgt[:, t:t + 1], cache)
            errs.append((last.float() - a[:, t]).abs().max().item())
        print(f"  cache vs logits max diff {max(errs):.2e} over {tgt_len} steps (should be < 1e-2)")
    return diff

In [ ]:
%%writefile pcr_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from sum_model import SumModel

BIDIRECTIONAL = ("prefixlm", "encdec")


class PCRModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.sm = SumModel(self.cfg)
        self.name = self.sm.name
        self._drop_unused()
        default_pool = "mean" if self.name in BIDIRECTIONAL else "mean_last"
        self.pool = self.cfg.get("pool", default_pool)
        if self.pool not in ("mean", "last", "first", "max", "mean_last"):
            raise ValueError(f"pool must be mean | last | first | max | mean_last, got {self.pool!r}")
        self.skip_prompt = bool(self.cfg.get("skip_prompt", True))
        d = self.sm.d * (2 if self.pool == "mean_last" else 1)
        p = int(self.cfg.get("proj_dim", 0) or 0)
        self.proj = None
        if p:
            self.proj = nn.Linear(d, p, bias=False)
            nn.init.orthogonal_(self.proj.weight)
        self.out_dim = p or d

    def _drop_unused(self):
        bb = self.sm.backbone
        if self.name == "encdec":
            bb.decoder = None
            bb.dec_ln = None
            self.sm.use_decoder = False
        hn = self.sm.head_name
        if hn is not None:
            head = getattr(bb, hn)
            if head.weight is not self.sm.emb.weight:
                for q in head.parameters():
                    q.requires_grad = False

    def states(self, ids):
        sm = self.sm
        x = sm._embed(ids)
        T = x.shape[1]
        sm._check_len(T, "prompt + chunk")
        if self.name == "prefixlm":
            h = sm._states_prefixlm(x, T)
        elif self.name == "encdec":
            h = sm._encode(x)
        elif self.name == "arch4":
            h = sm._states_arch4(x)
        else:
            h = sm._states_biscale(x)
        if self.skip_prompt and sm.n_prompt:
            h = h[:, sm.n_prompt:]
        return h

    def embed(self, ids):
        h = self.states(ids)
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            if self.pool == "mean":
                v = h.mean(1)
            elif self.pool == "last":
                v = h[:, -1]
            elif self.pool == "first":
                v = h[:, 0]
            elif self.pool == "max":
                v = h.amax(1)
            else:
                v = torch.cat([h.mean(1), h[:, -1]], dim=-1)
            if self.proj is not None:
                v = self.proj(v)
        return v

    def forward(self, ids):
        return self.embed(ids)

    def param_groups(self, base_lr):
        groups = self.sm.param_groups(base_lr)
        if self.proj is not None:
            groups.append({"params": list(self.proj.parameters()),
                           "lr": base_lr * float(self.cfg.get("proj_lr_mult", 1.0))})
        return groups


def build_pcr_model(model_cfg):
    return PCRModel(model_cfg)


@torch.no_grad()
def check_pcr_model(model_cfg, L=256, B=2, seed=0):
    m = PCRModel(model_cfg).eval()
    dev = m.sm.emb.weight.device
    g = torch.Generator().manual_seed(seed)
    ids = torch.randint(1000, m.sm.vocab, (B, L), generator=g).to(dev)
    v = m.embed(ids)
    n_all = sum(q.numel() for q in m.parameters())
    n_tr = sum(q.numel() for q in m.parameters() if q.requires_grad)
    print(f"{model_cfg.get('phase1')}/{model_cfg.get('method')} pool={m.pool}: embed {tuple(v.shape)} | "
          f"trainable {n_tr/1e6:.3f}M of {n_all/1e6:.2f}M ({100*n_tr/n_all:.2f}%)")
    if m.name in BIDIRECTIONAL or m.pool in ("last", "mean_last"):
        ids2 = ids.clone()
        ids2[:, -1] = (ids2[:, -1] + 1) % m.sm.vocab
        moved = (m.embed(ids2) - v).abs().max().item()
        print(f"  changing the last token moves the vector by {moved:.2e} (expect > 0)")
    return v


In [ ]:
# 8 — PLAN (rule 17.5: every setting below is yours). >= 6 experiments: 5 different adaptation techniques + KD.
LAYOUT = "ONE"
T = dict(epochs=6, epoch_queries=None, lr=2e-4, batch_size=4, grad_accum=2, warmup_frac=0.1, weight_decay=0.01,
         schedule="linear", max_minutes=100, loss="infonce", tau=0.05, n_neg=7, neg_source="mix", hard_pool=50,
         in_batch=True, select_metric="dev_f1", eval_every=1, chunk_batch=16)
D = dict(chunk_len=512, n_chunks=4, chunk_mode="head_tail", train_chunks=2, agg="max", sim="cos")


M_FULL    = dict(phase1="prefixlm", method="full", backbone_lr_mult=0.1, grad_ckpt=True)
M_PARTIAL = dict(phase1="arch4", method="partial", unfreeze_layers=3, backbone_lr_mult=0.3)
M_LORMA   = dict(phase1="encdec", method="lorma", rank=16, alpha=32, lorma_variant="plus")
M_PREFIX  = dict(phase1="encdec", method="prefix", n_prefix=16, peft_lr_mult=10)
M_PROMPT  = dict(phase1="biscale", method="prompt", n_prompt=16, peft_lr_mult=10, backbone_dropout=0.0, grad_ckpt=True)
M_KD      = dict(phase1="prefixlm", method="full", keep_layers=8, keep_mode="spread",
                 backbone_lr_mult=0.1, grad_ckpt=True)
KD = dict(teacher="p1_full", alpha=0.5, kd_tau=0.05)

job = lambda name, m, **kw: dict(name=name, model=m, train=kw.pop("train", T), data=kw.pop("data", D), **kw)
QUEUES = {
    "ONE": {0: [job("p1_full", M_FULL), job("p6_kd_student", M_KD, kd=KD), job("p4_prefix", M_PREFIX)],
            1: [job("p2_partial", M_PARTIAL), job("p3_lorma", M_LORMA), job("p5_prompt", M_PROMPT)]},
}
PLAN = QUEUES[LAYOUT]
todo = [(j["name"], k) for js in PLAN.values() for j in js for part in (j["model"], j["train"], j.get("data") or {}, j.get("kd") or {})
        for k, v in part.items() if v == TODO]
assert not todo, f"fill these first: {todo}"
print("plan:", {g: [j["name"] for j in js] for g, js in PLAN.items()})

In [ ]:
# 9 — model check for every planned model at its REAL training size: embed shape, trainable params, gradients, peak memory
import importlib, pcr_model; importlib.reload(pcr_model)
for js in PLAN.values():
    for j in js:
        try:
            d, tr = j["data"], j["train"]
            n_docs = int(tr["batch_size"]) * (2 + int(tr["n_neg"]))
            n_rows = n_docs * int(d.get("train_chunks") or d["n_chunks"])
            L_ = int(d["chunk_len"])
            m = pcr_model.build_pcr_model(dict(j["model"])).cuda().train()
            torch.cuda.reset_peak_memory_stats()
            outs = []
            for i in range(0, n_rows, int(tr.get("chunk_batch", 64))):
                x = torch.randint(5, V, (min(int(tr.get("chunk_batch", 64)), n_rows - i), L_), device="cuda")
                with torch.autocast("cuda", dtype=torch.float16):
                    outs.append(m.embed(x).float())
            e = torch.cat(outs)
            assert e.dim() == 2 and e.shape[0] == n_rows, f"embed must return [B, D], got {tuple(e.shape)}"
            assert torch.isfinite(e).all(), "embed returned NaN/inf"
            torch.nn.functional.normalize(e, dim=-1).sum().backward()
            nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            tot, trn = sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)
            print(f"OK   {j['name']:18s} embed dim {e.shape[1]} | trainable {trn/1e6:.3f}M of {tot/1e6:.2f}M ({100*trn/tot:.2f}%) | "
                  f"one step = {n_docs} docs x {n_rows//n_docs} chunks x {L_} tokens: peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB | no-grad {nog[:3]}", flush=True)
            del m, e, outs
        except Exception as ex:
            print(f"FAIL {j['name']:18s} {type(ex).__name__}: {str(ex)[:300]}", flush=True)
        torch.cuda.empty_cache()

In [ ]:
# 10 — RUN THE PLAN (blocks until every queue is done; do NOT stop this cell while jobs are queued)
run_queues(PLAN, deadline_h=5.75, reserve_min=15)
summary([j["name"] for js in PLAN.values() for j in js])

In [ ]:
# 11 — results table (report) + download links + small backup zip (no weights)
import pandas as pd
from IPython.display import FileLink, display
rows = []
for rp in sorted(glob.glob(f"{R}/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke") or r["run"].endswith("_teacher"): continue
    v = r.get("dev", {}); h = r.get("history", [])
    rows.append(dict(run=r["run"], dev_f1_at7=v.get("f1"), dev_p_at7=v.get("p"), dev_r_at7=v.get("r"), dev_r50=v.get("r50"),
                     dev_r100=v.get("r100"), dev_f1_epoch0=(h[0].get("dev_f1") if h and h[0].get("epoch") == 0 else None),
                     trainable_M=r["params_trainable"]/1e6, total_M=r["params_total"]/1e6, train_min=r["train_minutes"],
                     peak_GB=r["peak_mem_gb"], epochs_done=len([x for x in h if x.get("epoch", 0) > 0]),
                     dev_eval_s=r.get("dev_eval_seconds"), test_s=r.get("test_seconds")))
df = pd.DataFrame(rows).sort_values("run"); display(df)
df.to_csv(f"{R}/results.csv", index=False)
os.chdir("/kaggle/working")
for n in df.run: display(FileLink(f"p7runs/{n}/{n}_submission.zip"))
display(FileLink("p7runs/results.csv"))
!zip -qr p7_backup.zip sum_model.py pcr_model.py p7runs -x "*.pt" "*smoke*"
display(FileLink("p7_backup.zip"))